# Ejemplo 3 · Pronosticar consumo con 3 modelos en 3 escenarios
**Estud-IA · Clase 5 · Intermedio.**

**Problema.** Tienes el consumo diario (energía, ventas, pedidos) y quieres estimar los próximos 28 días. Pero no todos los negocios se comportan igual, así que se comparan **3 modelos** en **3 escenarios** para ver **cuál modelo funciona en cuál situación**.

**Los 3 modelos, en palabras simples**
- **Promedio reciente (la vara de medir):** predice que cada día será como el promedio de las últimas 4 semanas, día de la semana por día de la semana. Si un modelo no le gana, no sirve.
- **Regresión lineal:** traza una línea que sigue la tendencia y ajusta un extra por día de la semana. Entiende «va subiendo», pero solo en línea recta.
- **Bosque aleatorio:** muchos árboles de preguntas («¿es fin de semana?», «¿cuánto fue hace 7 días?») que votan. Capta formas que no son rectas, pero **no sabe extrapolar**: no predice valores más altos que los que ya vio.

**Los 3 escenarios (datos ficticios generados aquí)**
1. **Estable:** consumo plano con ciclo semanal.
2. **Crecimiento:** el consumo sube todo el año.
3. **Ciclo mensual:** cada mes sube el consumo en la última semana (pago de quincena, cierre, campaña). Se repite, así que el pasado lo enseña.

**Regla de oro.** Se entrena con lo antiguo y se prueba con las **últimas 28 días** (el futuro que el modelo no vio). Nunca se separa al azar en series de tiempo.

## Parte 0 · Los 3 escenarios

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
rng = np.random.default_rng(1540)
N = 365; d = np.arange(N); dsem = d % 7
ciclo = np.where(dsem >= 5, -25, 10)

escenarios = {
 "1 · Estable":           200 + ciclo + rng.normal(0, 8, N),
 "2 · Crecimiento":       200 + 0.35 * d + ciclo + rng.normal(0, 8, N),
 "3 · Ciclo mensual":     200 + ciclo + np.where(d % 30 >= 23, 60, 0) + rng.normal(0, 8, N),
}
# Para usar TUS datos: una Serie por escenario o por producto, con un valor por día.

fig, ax = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for a, (nom, s) in zip(ax, escenarios.items()):
    a.plot(s, lw=0.8); a.axvline(N - 28, color="r", ls="--", lw=1); a.set_title(nom)
ax[0].set_ylabel("consumo diario"); plt.show()
print("La línea roja separa: izquierda = entrenamiento, derecha = últimas 28 días (prueba).")

## Parte 1 · Los 3 modelos
La IA arma las columnas de apoyo (día de la semana, consumo de hace 7 días, consumo de hace 14 días y la tendencia) y entrena cada modelo. Las columnas de apoyo usan solo el pasado, nunca el día que se predice.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

H = 28
def columnas(s):
    t = pd.DataFrame({"y": s})
    t["tendencia"] = np.arange(len(s)); t["dsem"] = t.tendencia % 7; t["dmes"] = t.tendencia % 30
    t["lag7"] = t.y.shift(7); t["lag14"] = t.y.shift(14)
    for j in range(7): t[f"d{j}"] = (t.dsem == j).astype(int)
    return t.dropna()

def predecir(nombre, s):
    t = columnas(s); ent, prueba = t.iloc[:-H], t.iloc[-H:]
    if nombre == "Promedio reciente":
        base = s[:-H][-28:]; med = {j: base[(np.arange(len(base)) + (len(s)-H-28)) % 7 == j].mean() for j in range(7)}
        return np.array([med[int(x)] for x in prueba.dsem]), prueba.y.values
    cols_lin = ["tendencia"] + [f"d{j}" for j in range(7)]
    cols_rf  = ["dsem", "dmes", "lag7", "lag14"]
    if nombre == "Regresión lineal":
        m = LinearRegression().fit(ent[cols_lin], ent.y); return m.predict(prueba[cols_lin]), prueba.y.values
    m = RandomForestRegressor(n_estimators=200, random_state=1540).fit(ent[cols_rf], ent.y)
    # el bosque usa lag7 y lag14 reales (disponibles al predecir hasta 7 días adelante)
    return m.predict(prueba[cols_rf]), prueba.y.values

## Parte 2 · Comparar: error promedio (MAE) por modelo y escenario
**MAE** = cuántas unidades se equivoca en promedio. Menor es mejor. Un error solo se entiende frente a la vara de medir.

In [ ]:
modelos = ["Promedio reciente", "Regresión lineal", "Bosque aleatorio"]
res = {}
for nom, s in escenarios.items():
    res[nom] = {m: round(mean_absolute_error(*predecir(m, s)), 1) for m in modelos}
tabla = pd.DataFrame(res).T
display(tabla)
print("Mejor modelo por escenario:"); print(tabla.idxmin(axis=1).to_string())

### Míralo: real contra predicho
La tabla dice cuánto se equivoca cada uno; la gráfica dice **cómo**.

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(13, 8), sharex=True)
for i, (nom, s) in enumerate(escenarios.items()):
    for j, m in enumerate(modelos):
        p, real = predecir(m, s)
        ax[i, j].plot(real, label="real", lw=1.2); ax[i, j].plot(p, label="predicho", lw=1.2, ls="--")
        ax[i, j].set_title(f"{nom} · {m}", fontsize=9)
ax[0, 0].legend(); plt.tight_layout(); plt.show()

## Parte 3 · Qué sacar de aquí
Lee la tabla antes de leer esta nota. Pregúntate: ¿qué modelo ganó en cada escenario?, ¿alguno **no le ganó** a la vara de medir?, ¿qué pasó con el bosque en crecimiento (recuerda que no extrapola) y con la regresión lineal en el ciclo mensual (solo ve rectas)?

La lección no es «el modelo X es el mejor», sino **«el mejor modelo depende del escenario»** y que un error solo se entiende contra una referencia simple.

## Parte 4 · Un solo cambio, y se vuelve a medir
Cambia **una** cosa (por ejemplo, agregar una columna de festivo, o usar el promedio de 8 semanas) y repite la tabla de la Parte 2. Anota antes cuánto tendría que mejorar para que adoptes el cambio.

In [ ]:
# Tu cambio: por ejemplo, el bosque con más información del pasado (lag21)
def bosque_con_lag21(s):
    t = columnas(s); t["lag21"] = pd.Series(s).shift(21).values[-len(t):]
    t = t.dropna(); ent, prueba = t.iloc[:-H], t.iloc[-H:]
    cols = ["dsem", "dmes", "lag7", "lag14", "lag21"]
    m = RandomForestRegressor(n_estimators=200, random_state=1540).fit(ent[cols], ent.y)
    return mean_absolute_error(prueba.y, m.predict(prueba[cols]))
cambio = pd.DataFrame({"Bosque antes": tabla["Bosque aleatorio"], "Bosque con lag21": {n: round(bosque_con_lag21(s), 1) for n, s in escenarios.items()}})
display(cambio)

## Parte 5 · Tu turno: tu ficha convertida en prompt
Ficha (ejemplo): *Camino B. Cambio: agregar festivos. Medida: MAE en las últimas 4 semanas contra el promedio reciente. Criterio: lo adopto si baja el MAE en al menos 10 %. Riesgo: pocos datos, cambios bruscos.*

Prompt para tu IA:
```
Tengo el consumo diario con las columnas fecha y consumo.
Hazme un notebook de Colab que compare un promedio reciente, una regresión lineal y un bosque aleatorio
en 3 escenarios (estable, creciente y con un ciclo mensual).
Separa por fecha (entrena con lo antiguo y prueba con las últimas 28 días), muestra el MAE de cada modelo
en una tabla y la gráfica real contra predicho. Si no tengo datos, genera un año ficticio por escenario.
Explica cada celda sin matemáticas. No cambies nada más de lo que te pedí.
```
Entrega: la tabla de errores de tu proyecto, qué modelo ganó en qué situación y el único cambio que probarías después.